In [13]:
import torch 
from dreamerv4uwm.datasets import ShardedHDF5Dataset
DATA_PATH = "/media/mim-server/5a9b3378-c509-41de-b07f-544b25e6a481/soar_data_sharded"
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
resolution = (256, 256)

In [14]:
import mediapy
from torch.nn.functional import interpolate

dataset = ShardedHDF5Dataset(
        data_dir=DATA_PATH,
        window_size=64,
        stride=1,
        split='train',
        train_fraction=0.9,
        split_seed=123,
    )

# Arbitrary index into that episode
batch = dataset[400]
# imgs = batch["image"][:,[2, 1, 0], :, :]  # (T, C, H, W)
imgs = batch["image"]  # (T, C, H, W)
actions = batch["action"] # (1, T, N_lat, D_lat)
# actions=torch.zeros_like(actions)
imgs = interpolate(imgs, resolution).to(device=device)[None] # resize to tokenizer resolution

def plotVideo(video):
    imgs = video.cpu().permute(0,2,3,1).to(torch.float32).numpy()*255
    imgs = imgs.astype('uint8')
    mediapy.show_video(imgs, fps=10)

plotVideo(imgs[0])

Train split: 1227302 windows from 27438 episodes


This browser does not support the video tag.


# Dreamer V4 Tokenizer

In [15]:
from dreamerv4uwm.models.utils import load_tokenizer
from hydra import initialize, compose
from omegaconf import OmegaConf
with initialize(version_base=None, config_path="../scripts/config"):
    cfg = compose(config_name="dynamics/pushT.yaml")

tokenizer_ckpt="/home/mim-server/projects/rooholla/dreamerV4-UWM/checkpoints/tokenizer_ckpts/soar.pt"
cfg.tokenizer_ckpt=tokenizer_ckpt
tokenizer = load_tokenizer(cfg, 'cuda', max_num_forward_steps=300)
tokenizer = tokenizer.eval().cuda()

In [55]:
import torch
with torch.no_grad():
    with torch.autocast(device_type='cuda', dtype=torch.bfloat16):
        latents = tokenizer.encode(imgs)
        tau = 0.6 
        latents_noisy = (1-tau) * torch.randn_like(latents) + tau * latents
        imgs_recon = tokenizer.decode(latents_noisy)


plotVideo(imgs_recon[0])
plotVideo(imgs[0])

This browser does not support the video tag.


This browser does not support the video tag.


## Stable Diffusion 3 VAE (image tokenizer)

Wan-VAE compresses video; SD3's VAE only compresses single images, so each
frame is encoded independently. The pipeline mirrors the SD3 example:

1. **Encode**: `(B, 3, H, W)` image -> `(B, 16, H/8, W/8)` latent (16 channels, 8x downsample).
2. **Patchify** (MMDiT, patch=2): latent -> `(B, N, 16*2*2 = 64)` tokens.
3. Reverse: unpatchify -> VAE decode -> reconstructed frames.

The `imgs` tensor from above is `(1, T, C, H, W)` in `[0, 1]`. We fold the
time axis into the batch axis, shift to `[-1, 1]`, run all `T` frames as a
single batch, then plot the reconstruction.


In [17]:
from diffusers import AutoencoderKL

# SD3 VAE is gated on HF — accept the license and run `huggingface-cli login` first.
sd3_vae = AutoencoderKL.from_pretrained(
    "stabilityai/stable-diffusion-3-medium-diffusers",
    subfolder="vae",
    torch_dtype=torch.float16,
).to(device).eval()

print("SD3 VAE loaded.",
      "scaling_factor =", sd3_vae.config.scaling_factor,
      "shift_factor =", sd3_vae.config.shift_factor)


Cannot initialize model with low cpu memory usage because `accelerate` was not found in the environment. Defaulting to `low_cpu_mem_usage=False`. It is strongly recommended to install `accelerate` for faster and less memory-intense model loading. You can do so with: 
```
pip install accelerate
```
.


SD3 VAE loaded. scaling_factor = 1.5305 shift_factor = 0.0609


In [18]:
def patchify_2d(latent, p=2):
    """(B, C, H, W) -> (B, N, C*p*p) where N = (H/p)*(W/p)."""
    B, C, H, W = latent.shape
    assert H % p == 0 and W % p == 0
    x = latent.reshape(B, C, H // p, p, W // p, p)
    x = x.permute(0, 2, 4, 1, 3, 5).contiguous()
    return x.reshape(B, (H // p) * (W // p), C * p * p)


def unpatchify_2d(tokens, h, w, C=16, p=2):
    """Inverse of patchify_2d."""
    B = tokens.shape[0]
    x = tokens.reshape(B, h, w, C, p, p)
    x = x.permute(0, 3, 1, 4, 2, 5).contiguous()
    return x.reshape(B, C, h * p, w * p)


In [47]:
# imgs: (1, T, C, H, W) in [0, 1]  -> per-frame batch (T, C, H, W) in [-1, 1]
frames = imgs[0] * 2.0 - 1.0
frames = frames.to(dtype=torch.float16)
print("Frames:", tuple(frames.shape), f"range=[{frames.min():.2f}, {frames.max():.2f}]")

scaling = sd3_vae.config.scaling_factor
shift = sd3_vae.config.shift_factor

# 1. Encode every frame independently.
with torch.no_grad():
    latent = sd3_vae.encode(frames).latent_dist.mode()
    latent_norm = (latent - shift) * scaling
print("Latent:", tuple(latent_norm.shape), "(16ch, /8 spatial)")

# 2. Patchify into MMDiT tokens.
B, C, h, w = latent_norm.shape
tokens = patchify_2d(latent_norm, p=2)
print("Tokens:", tuple(tokens.shape),
      f"({h//2}x{w//2} = {tokens.shape[1]} tokens of dim {tokens.shape[-1]} per frame)")

# 3. Round-trip back to a latent grid (lossless).
latent_back = unpatchify_2d(tokens, h // 2, w // 2, C=C, p=2)
assert torch.allclose(latent_back, latent_norm)

Frames: (64, 3, 256, 256) range=[-1.00, 1.00]
Latent: (64, 16, 32, 32) (16ch, /8 spatial)
Tokens: (64, 256, 64) (16x16 = 256 tokens of dim 64 per frame)


In [46]:
latent_for_decode = latent_back / scaling + shift
tau = 0.5
noise_latent = torch.randn_like(latent_for_decode)
latent_for_decode_noisy = tau * latent_for_decode + (1 - tau) * noise_latent

# 4. Decode (un-normalize first).
with torch.no_grad():
    recon_sd3 = sd3_vae.decode(latent_for_decode_noisy).sample

# Back to [0, 1] so plotVideo can render it.
recon_sd3 = (recon_sd3.clamp(-1, 1) / 2 + 0.5).clamp(0, 1)
print("Reconstruction:", tuple(recon_sd3.shape))
plotVideo(recon_sd3)

Reconstruction: (64, 3, 256, 256)


This browser does not support the video tag.


## Wan2.1 VAE (spatiotemporal tokenizer)

Wan-VAE is a 3D causal autoencoder that compresses both space and time:

- 16 latent channels
- 8x spatial downsample
- 4x temporal downsample (causal: first frame is encoded standalone, then groups of 4)

So `T` input frames -> `(T-1)/4 + 1` latent frames; Wan's defaults always use
`T = 4k+1`. Our window has `T = 64`, so we trim to **61** frames -> 16 latent
frames.

Tokenization in Wan's MMDiT uses `(1, 2, 2)` patches over the latent grid,
giving tokens of dim `16 * 1 * 2 * 2 = 64` -- same per-token size as SD3, but
each token covers a 1-frame x 2 x 2 latent volume instead of a single image patch.


In [35]:
from diffusers import AutoencoderKLWan

# Wan-VAE is fp16-fragile (NaNs in temporal blocks); keep it in fp32.
wan_vae = AutoencoderKLWan.from_pretrained(
    "Wan-AI/Wan2.1-T2V-1.3B-Diffusers",
    subfolder="vae",
    torch_dtype=torch.float32,
).to(device).eval()
wan_vae.enable_tiling()  # spatial tiling so encode/decode fit on small GPUs

# Wan stores per-channel latent stats (length-16 lists) for whitening.
latents_mean = torch.tensor(wan_vae.config.latents_mean, dtype=torch.float32,
                            device=device).view(1, 16, 1, 1, 1)
latents_std = torch.tensor(wan_vae.config.latents_std, dtype=torch.float32,
                           device=device).view(1, 16, 1, 1, 1)
print("Wan2.1 VAE loaded. latents_mean[0:3] =",
      [round(m, 3) for m in wan_vae.config.latents_mean[:3]])


Cannot initialize model with low cpu memory usage because `accelerate` was not found in the environment. Defaulting to `low_cpu_mem_usage=False`. It is strongly recommended to install `accelerate` for faster and less memory-intense model loading. You can do so with: 
```
pip install accelerate
```
.


Wan2.1 VAE loaded. latents_mean[0:3] = [-0.757, -0.709, -0.911]


In [36]:
def patchify_3d(latent, pt=1, ph=2, pw=2):
    """(B, C, T, H, W) -> (B, N, C*pt*ph*pw), plus the (nt, nh, nw) grid."""
    B, C, T, H, W = latent.shape
    assert T % pt == 0 and H % ph == 0 and W % pw == 0
    nt, nh, nw = T // pt, H // ph, W // pw
    x = latent.reshape(B, C, nt, pt, nh, ph, nw, pw)
    x = x.permute(0, 2, 4, 6, 1, 3, 5, 7).contiguous()
    return x.reshape(B, nt * nh * nw, C * pt * ph * pw), (nt, nh, nw)


def unpatchify_3d(tokens, nt, nh, nw, C=16, pt=1, ph=2, pw=2):
    """Inverse of patchify_3d."""
    B = tokens.shape[0]
    x = tokens.reshape(B, nt, nh, nw, C, pt, ph, pw)
    x = x.permute(0, 4, 1, 5, 2, 6, 3, 7).contiguous()
    return x.reshape(B, C, nt * pt, nh * ph, nw * pw)


In [42]:
# imgs: (1, T=64, C, H, W) in [0, 1]  ->  Wan wants (1, C, T=4k+1, H, W) in [-1, 1].
NUM_FRAMES = 32  # 4*15 + 1
video = imgs[:, :NUM_FRAMES].permute(0, 2, 1, 3, 4).contiguous()  # (1, C, T, H, W)
video = (video * 2.0 - 1.0).to(dtype=torch.float32)
print("Video:", tuple(video.shape), f"range=[{video.min():.2f}, {video.max():.2f}]")

# 1. Encode (with per-channel whitening).
with torch.no_grad():
    latent = wan_vae.encode(video).latent_dist.mode()
    latent_norm = (latent - latents_mean) / latents_std
print(f"Latent: {tuple(latent_norm.shape)}  "
      f"(T={video.shape[2]} -> t={latent_norm.shape[2]}, /8 spatial)")

# 2. Patchify into 3D MMDiT tokens.
tokens, (nt, nh, nw) = patchify_3d(latent_norm, pt=1, ph=2, pw=2)
print(f"Tokens: {tuple(tokens.shape)}  "
      f"(grid {nt} x {nh} x {nw} = {tokens.shape[1]} tokens of dim {tokens.shape[-1]})")

# 3. Lossless round-trip.
latent_back_norm = unpatchify_3d(tokens, nt, nh, nw, C=latent.shape[1])
assert torch.allclose(latent_back_norm, latent_norm)

Video: (1, 3, 32, 256, 256) range=[-1.00, 1.00]
Latent: (1, 16, 8, 32, 32)  (T=32 -> t=8, /8 spatial)
Tokens: (1, 2048, 64)  (grid 8 x 16 x 16 = 2048 tokens of dim 64)


In [ ]:
latent_for_decode = latent_back_norm * latents_std + latents_mean
tau = 0.9
noise_latent = torch.randn_like(latent_for_decode)
latent_for_decode_noisy = tau * latent_for_decode + (1 - tau) * noise_latent
# 4. Decode (un-whiten first).
with torch.no_grad():
    recon_wan = wan_vae.decode(latent_for_decode_noisy, return_dict=False)[0]

# Back to (T, C, H, W) in [0, 1] for plotVideo.
recon_wan = (recon_wan.clamp(-1, 1) / 2 + 0.5).clamp(0, 1)
recon_wan_TCHW = recon_wan[0].permute(1, 0, 2, 3)  # (T, C, H, W)
print("Reconstruction:", tuple(recon_wan_TCHW.shape))
plotVideo(recon_wan_TCHW)


Reconstruction: (29, 3, 256, 256)
Pixel MSE vs. original (first 32 frames): 0.02127


This browser does not support the video tag.
